# ⏰ Notebook 1: The Synchronous Problem

Why synchronous processing fails for long operations.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why sync processing has limits
- HTTP timeout constraints
- Resource utilization problems
- User experience impact

## 🔧 Setup

Before running anything, make sure:

1. Docker services are up: `docker compose up -d` from the lab root.
2. Dependencies are installed: `uv sync` from the lab root.
3. **Kernel is the lab `.venv`**: click the kernel picker in the top-right of this notebook and pick the interpreter under `.venv/bin/python`.
4. If you don't see it, reload the VS Code window: `Cmd+Shift+P` → *Reload Window*, then re-open the notebook.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from typing import Callable

print("✅ Ready to learn about sync processing limits!")

### 🌍 Real-world

Stripe used to answer webhooks synchronously and would time out on slow merchant endpoints — they now enqueue delivery and retry asynchronously.

## ⏰ The Timeout Problem

In [ ]:
print("⏰ HTTP Timeout Reality")
print("=" * 60)
print("""
Common timeout limits in real systems:
─────────────────────────────────────────────────────────────

LOAD BALANCERS:
  • AWS ALB:        60 seconds (default)
  • Nginx:          60 seconds (default)
  • Cloudflare:    100 seconds (enterprise)

WEB FRAMEWORKS:
  • Gunicorn:       30 seconds (default worker timeout)
  • uWSGI:          30 seconds (default)
  • Node.js:       120 seconds (default)

BROWSERS:
  • Chrome:        300 seconds
  • Firefox:       300 seconds
  • But users leave after 10 seconds!

─────────────────────────────────────────────────────────────
REALITY: If your operation takes > 30 seconds, it will likely fail.
""")

In [ ]:
print("🔬 Simulating Sync Request with Timeout")
print("=" * 60)

def slow_operation(duration: int) -> str:
    print(f"   Starting {duration}s operation...")
    for i in range(duration):
        time.sleep(1)
        print(f"   Progress: {i+1}/{duration}s", end="\r")
    print()
    return f"Completed after {duration} seconds"

def handle_request_with_timeout(operation: Callable, timeout: int):
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(operation)
        try:
            result = future.result(timeout=timeout)
            return {"status": "success", "result": result}
        except TimeoutError:
            return {"status": "timeout", "error": f"Request timed out after {timeout}s"}

print("\n📋 Test 1: 3-second operation with 5-second timeout")
result = handle_request_with_timeout(lambda: slow_operation(3), timeout=5)
print(f"   Result: {result['status']} {'✅' if result['status'] == 'success' else '❌'}")

print("\n📋 Test 2: 10-second operation with 5-second timeout")
result = handle_request_with_timeout(lambda: slow_operation(10), timeout=5)
print(f"   Result: {result['status']} {'✅' if result['status'] == 'success' else '❌'}")
if result['status'] == 'timeout':
    print(f"   Error: {result['error']}")

## 📊 Resource Problems

In [ ]:
print("📊 Worker Blocking Problem")
print("=" * 60)
print("""
SCENARIO: Web server with 10 worker threads
─────────────────────────────────────────────────────────────

Normal operation (100ms requests):
  • Each worker handles 10 requests/second
  • 10 workers = 100 requests/second capacity

What happens when 10 users request PDF generation (45s each)?

t=0s:   All 10 workers start PDF generation
        [PDF] [PDF] [PDF] [PDF] [PDF] [PDF] [PDF] [PDF] [PDF] [PDF]
        
t=1s:   100 new requests arrive... NO WORKERS AVAILABLE!
        Queue: [req] [req] [req] [req] ... (100 waiting)

t=45s:  PDF jobs complete, but 4,500 requests have been waiting!
        Most have timed out. Users are angry. 😡

─────────────────────────────────────────────────────────────
ONE slow operation blocks ALL other requests!
""")

In [ ]:
print("🔬 Simulating Worker Pool Exhaustion")
print("=" * 60)

class SyncWebServer:
    def __init__(self, num_workers: int):
        self.num_workers = num_workers
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.stats = {"completed": 0, "rejected": 0, "in_progress": 0}
    
    def handle_request(self, request_type: str, duration: float) -> dict:
        if self.stats["in_progress"] >= self.num_workers:
            self.stats["rejected"] += 1
            return {"status": "rejected", "reason": "No workers available"}
        
        self.stats["in_progress"] += 1
        time.sleep(duration)
        self.stats["in_progress"] -= 1
        self.stats["completed"] += 1
        return {"status": "completed"}

server = SyncWebServer(num_workers=4)

print("\n📊 Scenario: 4 workers, mix of fast and slow requests")
print("   Submitting 4 slow requests (2s each)...")

with ThreadPoolExecutor(max_workers=10) as client_pool:
    slow_futures = [client_pool.submit(server.handle_request, "pdf", 2) for _ in range(4)]
    time.sleep(0.1)
    
    print(f"   Workers in use: {server.stats['in_progress']}/{server.num_workers}")
    print("\n   Now submitting 6 fast requests (0.1s each)...")
    
    fast_futures = [client_pool.submit(server.handle_request, "api", 0.1) for _ in range(6)]
    
    for f in slow_futures:
        f.result()
    for f in fast_futures:
        f.result()

print(f"\n📊 Results:")
print(f"   Completed: {server.stats['completed']}")
print(f"   Rejected: {server.stats['rejected']}")
print("\n❌ Fast requests blocked by slow ones!")

## 😰 User Experience Impact

In [ ]:
print("😰 User Experience with Sync Processing")
print("=" * 60)
print("""
What users see during a 45-second sync request:

t=0s:   User clicks "Generate Report"
        [Loading spinner appears]

t=5s:   "Is it working?"
        [Still loading...]

t=15s:  "Did something break?"
        [User considers refreshing]

t=25s:  "This is taking forever..."
        [User refreshes page]
        → Now there are TWO identical jobs running!

t=30s:  [Connection timeout]
        "Error: Request timed out"
        → User's work is LOST (or is it?)

─────────────────────────────────────────────────────────────

PROBLEMS:
1. No progress feedback
2. No way to cancel
3. Refresh creates duplicates
4. Timeout loses work
5. User doesn't know if it succeeded
""")

## ✅ The Solution Preview

In [ ]:
print("✅ The Async Solution")
print("=" * 60)
print("""
Instead of waiting, return immediately with a job ID.

SYNC (Bad):
─────────────────────────────────────────────────────────────
POST /generate-report
... 45 seconds later ...
Response: {"pdf_url": "..."} (or timeout!)

ASYNC (Good):
─────────────────────────────────────────────────────────────
POST /generate-report
Response (100ms): {"job_id": "abc-123", "status": "pending"}

GET /jobs/abc-123
Response: {"status": "processing", "progress": 45}

GET /jobs/abc-123
Response: {"status": "completed", "pdf_url": "..."}

─────────────────────────────────────────────────────────────

USER EXPERIENCE:
✓ Immediate feedback (job accepted)
✓ Progress updates available
✓ Can navigate away and come back
✓ Notification when complete
✓ Retry button if it fails
""")

## 🧪 Quick Quiz

1. **Why do load balancers have timeout limits?**

2. **What happens when users refresh during a slow request?**

3. **Why can't you just increase the timeout to 10 minutes?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why load balancer timeouts:")
print("   - Prevent resource exhaustion")
print("   - Detect hung connections")
print("   - Free up connections for other requests")
print()
print("2. User refresh during slow request:")
print("   - Creates duplicate jobs")
print("   - Original job keeps running")
print("   - Wastes resources, confusing results")
print()
print("3. Why not just increase timeout:")
print("   - Workers stay blocked longer")
print("   - Memory/connections held longer")
print("   - Users won't wait that long anyway")
print("   - Doesn't solve the feedback problem")

## 📚 Summary

### Key Takeaways

1. **Timeouts are real** - 30-60 seconds is typical max
2. **Workers block** - Slow ops starve fast ops
3. **Users don't wait** - They refresh or leave
4. **No feedback** - Users don't know what's happening
5. **Solution: async** - Accept fast, process in background

### Next Up

In **Notebook 2**, we'll implement the queue:
- Redis as a job queue
- Submitting jobs
- Job status tracking